In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── Style ──────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted")
SEED = 42
np.random.seed(SEED)

# ── Load Data ──────────────────────────────────────────────
df = pd.read_csv('/mnt/user-data/uploads/diabetes.csv')
print(f"Population shape: {df.shape}")
print(df.describe().round(2))

# ══════════════════════════════════════════════════════════
# PART A — Random Sample of 25, Compare Glucose Statistics
# ══════════════════════════════════════════════════════════
np.random.seed(SEED)
sample_25 = df.sample(n=25, random_state=SEED)

pop_mean_gluc   = df['Glucose'].mean()
pop_max_gluc    = df['Glucose'].max()
samp_mean_gluc  = sample_25['Glucose'].mean()
samp_max_gluc   = sample_25['Glucose'].max()

print("\n" + "="*55)
print("PART A: Glucose — Sample (n=25) vs Population")
print("="*55)
print(f"  Population Mean Glucose : {pop_mean_gluc:.2f}")
print(f"  Sample     Mean Glucose : {samp_mean_gluc:.2f}")
print(f"  Population Max  Glucose : {pop_max_gluc:.2f}")
print(f"  Sample     Max  Glucose : {samp_max_gluc:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Part A — Glucose: Sample (n=25) vs Population", fontsize=14, fontweight='bold')

# Left: overlapping histograms
axes[0].hist(df['Glucose'], bins=25, alpha=0.55, color='steelblue', label=f'Population (n={len(df)})', edgecolor='white')
axes[0].hist(sample_25['Glucose'], bins=10, alpha=0.75, color='tomato', label='Sample (n=25)', edgecolor='white')
axes[0].axvline(pop_mean_gluc,  color='steelblue', linestyle='--', linewidth=2, label=f'Pop Mean: {pop_mean_gluc:.1f}')
axes[0].axvline(samp_mean_gluc, color='tomato',    linestyle='--', linewidth=2, label=f'Sample Mean: {samp_mean_gluc:.1f}')
axes[0].set_xlabel('Glucose'); axes[0].set_ylabel('Count')
axes[0].set_title('Distribution Comparison'); axes[0].legend()

# Right: bar chart comparing mean & max
stats      = ['Mean Glucose', 'Max Glucose']
pop_vals   = [pop_mean_gluc,  pop_max_gluc]
samp_vals  = [samp_mean_gluc, samp_max_gluc]
x = np.arange(len(stats)); w = 0.35
bars1 = axes[1].bar(x - w/2, pop_vals,  w, label='Population', color='steelblue', alpha=0.85)
bars2 = axes[1].bar(x + w/2, samp_vals, w, label='Sample',     color='tomato',    alpha=0.85)
axes[1].bar_label(bars1, fmt='%.1f', padding=3)
axes[1].bar_label(bars2, fmt='%.1f', padding=3)
axes[1].set_xticks(x); axes[1].set_xticklabels(stats)
axes[1].set_ylabel('Glucose Level')
axes[1].set_title('Mean & Max Comparison'); axes[1].legend()

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/partA_glucose_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print("  → Chart saved: partA_glucose_comparison.png")


# ══════════════════════════════════════════════════════════
# PART B — 98th Percentile of BMI: Sample vs Population
# ══════════════════════════════════════════════════════════
pop_98_bmi  = np.percentile(df['Glucose'].values, 98)   # keep Glucose scope for context
pop_98_bmi  = np.percentile(df['BMI'].values, 98)
samp_98_bmi = np.percentile(sample_25['BMI'].values, 98)

print("\n" + "="*55)
print("PART B: BMI 98th Percentile — Sample vs Population")
print("="*55)
print(f"  Population 98th pct BMI : {pop_98_bmi:.2f}")
print(f"  Sample     98th pct BMI : {samp_98_bmi:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Part B — BMI 98th Percentile: Sample (n=25) vs Population", fontsize=14, fontweight='bold')

# Left: overlapping KDE + percentile lines
sns.kdeplot(df['BMI'], ax=axes[0], fill=True, color='steelblue', alpha=0.4, label=f'Population (n={len(df)})')
sns.kdeplot(sample_25['BMI'], ax=axes[0], fill=True, color='tomato', alpha=0.5, label='Sample (n=25)')
axes[0].axvline(pop_98_bmi,  color='steelblue', linestyle='--', linewidth=2, label=f'Pop 98th pct: {pop_98_bmi:.1f}')
axes[0].axvline(samp_98_bmi, color='tomato',    linestyle='--', linewidth=2, label=f'Sample 98th pct: {samp_98_bmi:.1f}')
axes[0].set_xlabel('BMI'); axes[0].set_ylabel('Density')
axes[0].set_title('BMI Distribution & 98th Percentile'); axes[0].legend()

# Right: bar comparison
cats = ['Population\n98th pct BMI', 'Sample\n98th pct BMI']
vals = [pop_98_bmi, samp_98_bmi]
colors = ['steelblue', 'tomato']
bars = axes[1].bar(cats, vals, color=colors, alpha=0.85, width=0.4)
axes[1].bar_label(bars, fmt='%.2f', padding=4, fontsize=11)
axes[1].set_ylabel('BMI'); axes[1].set_ylim(0, max(vals)*1.2)
axes[1].set_title('98th Percentile BMI Comparison')

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/partB_bmi_percentile.png', dpi=150, bbox_inches='tight')
plt.close()
print("  → Chart saved: partB_bmi_percentile.png")


# ══════════════════════════════════════════════════════════
# PART C — Bootstrap: 500 samples of 150, BloodPressure
# ══════════════════════════════════════════════════════════
print("\n" + "="*55)
print("PART C: Bootstrap — BloodPressure (500 samples × 150 obs)")
print("="*55)

N_SAMPLES, SAMPLE_SIZE = 500, 150
boot_means = np.zeros(N_SAMPLES)
boot_stds  = np.zeros(N_SAMPLES)
boot_p50   = np.zeros(N_SAMPLES)   # median
boot_p98   = np.zeros(N_SAMPLES)

for i in range(N_SAMPLES):
    s = df['BloodPressure'].sample(n=SAMPLE_SIZE, replace=True, random_state=SEED + i)
    boot_means[i] = s.mean()
    boot_stds[i]  = s.std()
    boot_p50[i]   = np.percentile(s, 50)
    boot_p98[i]   = np.percentile(s, 98)

# Population stats
pop_bp_mean = df['BloodPressure'].mean()
pop_bp_std  = df['BloodPressure'].std()
pop_bp_p50  = np.percentile(df['BloodPressure'], 50)
pop_bp_p98  = np.percentile(df['BloodPressure'], 98)

# Bootstrap averages
boot_avg_mean = boot_means.mean()
boot_avg_std  = boot_stds.mean()
boot_avg_p50  = boot_p50.mean()
boot_avg_p98  = boot_p98.mean()

print(f"  {'Statistic':<28} {'Population':>12} {'Bootstrap Avg':>14}")
print(f"  {'-'*55}")
print(f"  {'Mean BloodPressure':<28} {pop_bp_mean:>12.2f} {boot_avg_mean:>14.2f}")
print(f"  {'Std Dev BloodPressure':<28} {pop_bp_std:>12.2f} {boot_avg_std:>14.2f}")
print(f"  {'50th Percentile (Median)':<28} {pop_bp_p50:>12.2f} {boot_avg_p50:>14.2f}")
print(f"  {'98th Percentile':<28} {pop_bp_p98:>12.2f} {boot_avg_p98:>14.2f}")

# ── Chart C1: Bootstrap distributions with pop reference lines
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Part C — Bootstrap BloodPressure (500×150) vs Population", fontsize=14, fontweight='bold')

metrics = [
    (boot_means, pop_bp_mean,  boot_avg_mean, 'Bootstrap Sample Means',        'Mean BP'),
    (boot_stds,  pop_bp_std,   boot_avg_std,  'Bootstrap Sample Std Devs',      'Std Dev BP'),
    (boot_p50,   pop_bp_p50,   boot_avg_p50,  'Bootstrap Sample Medians (P50)', 'Median BP'),
    (boot_p98,   pop_bp_p98,   boot_avg_p98,  'Bootstrap Sample P98',           '98th Pct BP'),
]

for ax, (boot_vals, pop_val, boot_avg, title, xlabel) in zip(axes.flat, metrics):
    ax.hist(boot_vals, bins=30, color='mediumseagreen', edgecolor='white', alpha=0.8, label='Bootstrap dist.')
    ax.axvline(pop_val,   color='crimson',    linewidth=2.2, linestyle='--', label=f'Population: {pop_val:.2f}')
    ax.axvline(boot_avg,  color='darkorange', linewidth=2.2, linestyle='-',  label=f'Boot Avg: {boot_avg:.2f}')
    ax.set_title(title, fontweight='bold'); ax.set_xlabel(xlabel); ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/partC_bootstrap_distributions.png', dpi=150, bbox_inches='tight')
plt.close()

# ── Chart C2: Side-by-side summary bar chart
fig, ax = plt.subplots(figsize=(11, 5))
fig.suptitle("Part C — Population vs Bootstrap Average: BloodPressure Stats", fontsize=13, fontweight='bold')

labels   = ['Mean', 'Std Dev', 'Median (P50)', '98th Pct']
pop_vals = [pop_bp_mean, pop_bp_std, pop_bp_p50, pop_bp_p98]
bt_vals  = [boot_avg_mean, boot_avg_std, boot_avg_p50, boot_avg_p98]
x = np.arange(len(labels)); w = 0.35

b1 = ax.bar(x - w/2, pop_vals, w, label='Population', color='steelblue', alpha=0.85)
b2 = ax.bar(x + w/2, bt_vals,  w, label='Bootstrap Avg (500×150)', color='mediumseagreen', alpha=0.85)
ax.bar_label(b1, fmt='%.2f', padding=3, fontsize=9)
ax.bar_label(b2, fmt='%.2f', padding=3, fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('BloodPressure Value'); ax.legend()

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/partC_summary_comparison.png', dpi=150, bbox_inches='tight')
plt.close()

print("  → Charts saved: partC_bootstrap_distributions.png, partC_summary_comparison.png")
print("\nAll done!")
